## Import package

In [1]:
import sys
sys.path.append('./script')
# import ETGEMs_function_protain as etgf
from pyomo_solving import *
from ET_optme import *
import pandas as pd
import cobra
import gurobipy
import json
import multiprocessing
import os
from multiprocessing import Pool
from tqdm import tqdm 
from sympy import subsets
import pandas as pd
import matplotlib.pyplot as plt
import re
from ET_optme import _find_feasible_product_coeff, _find_feasible_biomass_coeff
# data
model_file="./file/iCZ870.json" # c.glutamicum model, irreversible and reactions with isoezyme splited
reaction_kcat_MW_file='./file/reaction_kcat_MW_modify_n.csv'
# results
path_results = './results/icz870_lysine'
savepath = './results/icz870_picture'
dictionarymodel_path='./results/etcz870.json' # dictionary model saving path

wild_enzyme_concentration_path = './results/ET-EComp/icz870_lysine/wild-enzyme.json'
over_enzyme_concentration_path = './results/ET-EComp/icz870_lysine/over-enzyme.json'

# case study
model0_file = './file/iCZ870.json'
reaction_g0_file=os.path.join('./file/iCZ870_g0_n.csv') #  standard Gibbs free energy files
metabolites_lnC_file = os.path.join('file/iCZ870_met.txt') # concentration range files
# model、substrate、product、oxygence
# mode: SET-->Enzyme-Thermo constrained, mode: SE-->Enzyme constrained, mode: ST-->Thermo constrained,mode: S-->Stoichiometric constrained
inputdic = {"model":'iCZ870.json',
"substrate":"EX_cpd00027_e_reverse",
"biomass": "EX_biomass_c",
"product": "EX_cpd00054_e",
"taskname": "optforce",
"mode":"SET",
"oxygenstate":"aerobic"}
# save task.json
with open('./results/task.json', 'w') as json_file:
    json.dump(inputdic, json_file, indent=4)
path_strain = 'iCW'

## 1.Enzyme-thermo model construction

In [2]:
# read model
model = cobra.io.load_json_model(model_file)

# Store model data into Get_Concretemodel_Need_Data
Concretemodel_Need_Data = Get_Concretemodel_Need_Data(model_file)

model0 = cobra.io.load_json_model(model0_file)

# Save the protein-centered model as a dictionary
dictionary_model = json_load(dictionarymodel_path)
dictionary_model.keys()

rname3 = []

# Store the protein-centered dictionary into Get_Concretemodel_Need_Data
get_dictionarymodel_data2(dictionary_model, Concretemodel_Need_Data, rname3)

# Store thermodynamic data into Get_Concretemodel_Need_Data
Get_Concretemodel_Need_Data_g0(Concretemodel_Need_Data, reaction_g0_file, metabolites_lnC_file, reaction_kcat_MW_file)

Concretemodel_Need_Data['reaction_g0']['g0'] = Concretemodel_Need_Data['reaction_g0']['g0'].replace(0, np.nan)
Concretemodel_Need_Data['reaction_g0'].dropna(subset=['g0'], inplace=True)

# Concretemodel_Need_Data['reaction_g0'].at['AIRC3_reverse', 'g0'] = 0
# Concretemodel_Need_Data['reaction_g0'].at['ATPS4rpp_reverse_num2', 'g0'] = 0

Inc = Concretemodel_Need_Data['metabolites_lnC']

for i in Concretemodel_Need_Data['metabolite_list']:
    if i not in Inc.index:
        Inc.loc[i, 'lnClb'] = -14.508658
        Inc.loc[i, 'lnCub'] = -3.912023

Concretemodel_Need_Data['metabolites_lnC'] = Inc

# Modify the oxygen conditions of the model
if inputdic['oxygenstate'] == 'aerobic':
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 1000
if inputdic['oxygenstate'] == 'micro_aerobic': 
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 2
if inputdic['oxygenstate'] == 'anaerobic': 
    model.reactions.get_by_id('EX_cpd00007_e_reverse').upper_bound = 0

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2788620
Academic license 2788620 - for non-commercial use only - registered to 90___@qq.com


## 2.Algorithm :  ET-EComp

In [3]:
# Calculate the maximum growth rate
B_value1,v0_biomass,bio,totalE1,objvalue2 = calculate_biomass(Concretemodel_Need_Data,inputdic,model,'iCW')# Calculate the enzyme

maximum growth rate value is: 0.0(mmol/gDW/h)
The maximum thermodynamic driving force value is : 0.0
minimizing the total enzyme concentration  is: 0.0007942524718070124((mmol/gDW)


# 固定系数

In [ ]:
# Calculate the enzyme concentration range under maximum growth conditions
enzyme_list = list(Concretemodel_Need_Data['mw_dict'].keys())
obj_names = enzyme_list
pool = multiprocessing.Pool(processes=multiprocessing.cpu_count())

# Use multiprocessing to calculate the wild range for each enzyme
results = pool.starmap(calculate_wildrange, [(Concretemodel_Need_Data, obj_name, inputdic, objvalue2, v0_biomass, totalE1, path_strain, B_value1) for obj_name in obj_names])

pool.close()
pool.join()

final_results = {}
for result in results:
    final_results.update(result)

# 保存野生型结果
os.makedirs(path_results, exist_ok=True)
wild_file_path = os.path.join(path_results, 'wild-enzyme.json')
with open(wild_file_path, 'w') as f:
    json.dump(final_results, f, indent=4)
print(f"野生型酶浓度范围已保存至: {wild_file_path}")


In [ ]:
# Calculate the maximum production rate
B_value2,v1_product_max,pro,totalE2,EcoECM_FBA_protainmodel_B2 = calculate_product(Concretemodel_Need_Data,inputdic,objvalue2,path_strain,v0_biomass)# The enzymes that catalyze each reaction in the model
enzyme_list = list(Concretemodel_Need_Data['mw_dict'].keys())
obj_names = enzyme_list
pool = multiprocessing.Pool(processes=multiprocessing.cpu_count())

results = pool.starmap(calculate_over, [(Concretemodel_Need_Data,obj_name,inputdic,objvalue2,v0_biomass,v1_product_max,totalE2,path_strain,B_value2) for obj_name in obj_names])
pool.close()
pool.join()

final_results = {}
for result in results:
    final_results.update(result)

over_file_path = os.path.join(path_results, 'over-enzyme.json')
with open(over_file_path, 'w') as f:
    json.dump(final_results, f, indent=4)
print(f"过表达酶浓度范围已保存至: {over_file_path}")

# 自动确定系数

In [ ]:
import copy
# concentration range under maximum growth conditions
enzyme_list = list(Concretemodel_Need_Data['mw_dict'].keys())
obj_names = enzyme_list
if inputdic['mode'] == 'SET':
    test_genes = list(Concretemodel_Need_Data['enzyme_rxns_dict'].keys())[:5]
    Concretemodel_Need_Data['E_total'] = totalE1
    Concretemodel_Need_Data['B_value'] = B_value1 * 0.98
    biomass_coeff = _find_feasible_biomass_coeff(
        Concretemodel_Need_Data,
        {'fix_reactions': {},
         'substrate_constrain': (inputdic['substrate'], objvalue2)},
        inputdic, v0_biomass, test_genes)
    if biomass_coeff is None:
        raise ValueError("[SET] biomass_constrain 系数从0.95降至0.90仍无解")
else:
    biomass_coeff = None

# ★ 修复1：在此处加deepcopy
clean_data_wild = copy.deepcopy(Concretemodel_Need_Data)

pool = multiprocessing.Pool(processes=multiprocessing.cpu_count())
results = pool.starmap(calculate_wildrange, [
    (clean_data_wild, obj_name, inputdic, objvalue2,  # ← 改这里
     v0_biomass, totalE1, path_strain, B_value1, biomass_coeff)
    for obj_name in obj_names
])
pool.close()
pool.join()

final_results = {}
for result in results:
    final_results.update(result)

os.makedirs(path_results, exist_ok=True)
wild_file_path = os.path.join(path_results, 'wild-enzyme.json')
with open(wild_file_path, 'w') as f:
    json.dump(final_results, f, indent=4)
print(f"野生型酶浓度范围已保存至: {wild_file_path}")

B_value2, v1_product_max, pro, totalE2, EcoECM_FBA_protainmodel_B2 = calculate_product(
    Concretemodel_Need_Data, inputdic, objvalue2, path_strain, v0_biomass
)

enzyme_list = list(Concretemodel_Need_Data['mw_dict'].keys())
obj_names = enzyme_list

if inputdic['mode'] == 'SET':
    test_genes = list(Concretemodel_Need_Data['enzyme_rxns_dict'].keys())[:3]
    Concretemodel_Need_Data['E_total'] = totalE2 * 1.01
    Concretemodel_Need_Data['B_value'] = B_value2 * 0.99
    product_coeff = _find_feasible_product_coeff(
        Concretemodel_Need_Data,
        {'fix_reactions': {},
         'substrate_constrain': (inputdic['substrate'], objvalue2),
         'biomass_constrain': (inputdic['biomass'], v0_biomass * 0.1)},
        inputdic, v1_product_max, test_genes)
    if product_coeff is None:
        raise ValueError("[SET] product_constrain 系数从0.95降至0.90仍无解")
else:
    product_coeff = None

# ★ 修复2：在此处加deepcopy
clean_data_over = copy.deepcopy(Concretemodel_Need_Data)

pool = multiprocessing.Pool(processes=multiprocessing.cpu_count())
results = pool.starmap(calculate_over, [
    (clean_data_over, obj_name, inputdic, objvalue2,  # ← 改这里
     v0_biomass, v1_product_max, totalE2, path_strain, B_value2, product_coeff)
    for obj_name in obj_names
])
pool.close()
pool.join()

final_results = {}
for result in results:
    final_results.update(result)

over_file_path = os.path.join(path_results, 'over-enzyme.json')
with open(over_file_path, 'w') as f:
    json.dump(final_results, f, indent=4)
print(f"过表达酶浓度范围已保存至: {over_file_path}")

model.name="unknown";
    - termination condition: infeasible
    - message from solver: Model was proven to be infeasible.
ERROR: evaluating object as numeric value: e1[CEY17_RS00005 and CEY17_RS00015
and CEY17_RS00030 and CEY17_RS00065 and CEY17_RS01420 and CEY17_RS01795 and
CEY17_RS01810 and CEY17_RS04650 and CEY17_RS06755 and CEY17_RS07650 and
CEY17_RS07665 and CEY17_RS08565 and CEY17_RS09885 and CEY17_RS10135 and
CEY17_RS10470 and CEY17_RS11520 and CEY17_RS12185 and CEY17_RS12430 and
CEY17_RS13425 and CEY17_RS15525]
        (object: <class 'pyomo.core.base.var.VarData'>)
    No value for uninitialized NumericValue object e1[CEY17_RS00005 and
    CEY17_RS00015 and CEY17_RS00030 and CEY17_RS00065 and CEY17_RS01420 and
    CEY17_RS01795 and CEY17_RS01810 and CEY17_RS04650 and CEY17_RS06755 and
    CEY17_RS07650 and CEY17_RS07665 and CEY17_RS08565 and CEY17_RS09885 and
    CEY17_RS10135 and CEY17_RS10470 and CEY17_RS11520 and CEY17_RS12185 and
    CEY17_RS12430 and CEY17_RS13425 and C

## 3. process results

In [ ]:
path_results = './results/icz870_lysine'

# Read results file
enzyme_results_data, enzyme_overresults2_data = read_file(path_results)

# Filter the targets that need modification
ko_data, up_data, down_data = compare_results(enzyme_results_data, enzyme_overresults2_data, inputdic)

# get reaction equation
reaction_list = Concretemodel_Need_Data['e0']
equation_dict = gene_reaction_map1(reaction_list, model)

# Enzyme concentrations and flux distribution under the wild-type condition
E_refdict, mw_dict, reaction_dict_bio, kcat_dict = ref_e_con(inputdic, Concretemodel_Need_Data, totalE1, path_strain,
                                                             objvalue2)

# Retrieve gene, equation, and reaction mappings
new_gene_reaction_mapping_bio, gene_reaction_mapping, gene_kcat_mapping = gene_reaction_map(enzyme_list,
                                                                                            reaction_dict_bio,
                                                                                            kcat_dict,
                                                                                            Concretemodel_Need_Data,
                                                                                            model)

# Retrieve the reaction flux distribution under maximum production conditions
reaction_dict, kcat_dict, EcoECM_FBA_protainmodel_pro_max = reaction_flux(inputdic, Concretemodel_Need_Data, totalE2,
                                                                          path_strain, totalE1, objvalue2, v0_biomass)

# Retrieve the enzyme concentrations and corresponding reaction flux distributions for the overexpressed state
new_gene_reaction_mapping, gene_reaction_mapping, gene_kcat_mapping = gene_reaction_map(enzyme_list, reaction_dict,
                                                                                        kcat_dict,
                                                                                        Concretemodel_Need_Data, model)

# Retrieve the enzyme usage proportions for each enzyme in the model
normalized_E_dict, E_dict, e_dict = enzyme_usage_over(EcoECM_FBA_protainmodel_pro_max, Concretemodel_Need_Data)

# Calculate the adjustment level for each modification strategy
Fold_change = fold_change(E_dict, E_refdict)

# List of all reactions in the model
reaction_list = [rea.id for rea in model.reactions]

# Retrieve the reaction equations
equation_dict = gene_reaction_map1(reaction_list, model)

# Generate the result table
meged_df = must_df(inputdic, equation_dict, new_gene_reaction_mapping_bio, new_gene_reaction_mapping, E_refdict, E_dict,
                   Fold_change, normalized_E_dict, e_dict, gene_kcat_mapping, enzyme_results_data,
                   enzyme_overresults2_data, ko_data, up_data, down_data, Concretemodel_Need_Data)

# Overflow reactions in the model for wild and overexpressed states
bio_df, pro_df, KeyInfo = basic(bio, pro, B_value1, v0_biomass, B_value2, v1_product_max, totalE1, totalE2, inputdic)

# Identify bottleneck reactions under thermodynamic constraints
bottleneck_reactions_list, Df = bottleneck_reactions(EcoECM_FBA_protainmodel_B2, model)

# Generate the result table
output(KeyInfo, path_results, bio_df, pro_df, meged_df, inputdic, Df)


## Analyze ET-OptForce results
## Case Studies

In [ ]:
lysine_S = pd.read_excel('./results/ET-EComp/lysine/S_lys/results.xlsx',sheet_name='MUST')
lysine_ST = pd.read_excel('./results/ET-EComp/lysine/T_lys/results.xlsx',sheet_name='MUST')
lysine_SE = pd.read_excel('./results/ET-EComp/lysine/E_lys/results.xlsx',sheet_name='MUST')
lysine_SET = pd.read_excel('./results/ET-EComp/lysine/ET_lys/results.xlsx',sheet_name='MUST')

In [ ]:
lysine_SET = lysine_SET.drop('Unnamed: 0', axis=1)
lysine_S = lysine_S.drop('Unnamed: 0', axis=1)
lysine_ST = lysine_ST.drop('Unnamed: 0', axis=1)
lysine_SE = lysine_SE.drop('Unnamed: 0', axis=1)

In [ ]:
pyrc = lysine_S[lysine_S['reaction']=='PYRC']
ASPK = lysine_S[lysine_S['reaction']=='ASPK']
IPPS = lysine_S[lysine_S['reaction']=='IPPS']
GAPD = lysine_S[lysine_S['reaction']=='GAPD_num1']

In [ ]:
pyrc = pyrc.iloc[:, :4]
ASPK = ASPK.iloc[:, :4]
IPPS = IPPS.iloc[:, :4]
GAPD = GAPD.iloc[:, :4]
S_result_pyrc = {}
S_result_aspk = {}
S_result_ipps = {}
S_result_gapd = {}
for index, row in pyrc.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    s_temp_dict_pyrc = {"id":"OptForce","reaction": 'PYRC:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    S_result_pyrc[reaction_name] = s_temp_dict_pyrc
for index, row in ASPK.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    s_temp_dict_aspk = {"id":"OptForce","reaction":'ASPK:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    S_result_aspk[reaction_name] = s_temp_dict_aspk

for index, row in IPPS.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    s_temp_dict_ipps = {"id":"OptForce","reaction": 'IPPS:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    S_result_ipps[reaction_name] = s_temp_dict_ipps
for index, row in GAPD.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    s_temp_dict_gapd = {"id":"OptForce","reaction": 'GAPD:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    S_result_gapd[reaction_name] = s_temp_dict_gapd

In [ ]:
tdf_pyrc = lysine_ST[lysine_ST['reaction']=='PYRC']
tdf_aspk = lysine_ST[lysine_ST['reaction']=='ASPK']
tdf_ipps = lysine_ST[lysine_ST['reaction']=='IPPS']
tdf_gapd = lysine_ST[lysine_ST['reaction']=='GAPD']

In [ ]:


tdf_pyrc = tdf_pyrc.iloc[:, :4]
tdf_aspk = tdf_aspk.iloc[:, :4]
tdf_ipps = tdf_ipps.iloc[:, :4]
tdf_gapd = tdf_gapd.iloc[:, :4]
T_result_pyrc = {}
T_result_aspk = {}
T_result_ipps = {}
T_result_gapd = {}
for index, row in tdf_pyrc.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    T_result_dict_pyrc = {"id":"T_OptForce","reaction":'PYRC:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    T_result_pyrc[reaction_name] = T_result_dict_pyrc
for index, row in tdf_aspk.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    T_result_dict_aspk = {"id":"T_OptForce","reaction":'ASPK:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    T_result_aspk[reaction_name] = T_result_dict_aspk

for index, row in tdf_ipps.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    T_result_dict_ipps = {"id":"T_OptForce","reaction":'IPPS:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    T_result_ipps[reaction_name] = T_result_dict_ipps
for index, row in tdf_gapd.iterrows():
    reaction_name = row['reaction']
    equation = row['equation']
    flux_wild_values = [float(value.strip("'")) for value in row['flux_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['flux_over'].strip('[]').split(', ')]
    T_result_dict_gapd = {"id":"T_OptForce","reaction":'GAPD:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    T_result_gapd[reaction_name] = T_result_dict_gapd

In [ ]:
ecdf_pyrc = lysine_SE[lysine_SE['reaction']=='PYRC']
ecdf_aspk = lysine_SE[lysine_SE['reaction']=='ASPK']
ecdf_ipps = lysine_SE[lysine_SE['reaction']=='IPPS']
ecdf_gapd = lysine_SE[lysine_SE['gene']=='Cgl0937']

In [ ]:
E_result_dict_pyrc = {}
E_result_dict_aspk = {}
E_result_dict_ipps = {}
E_result_dict_gapd = {}


for index, row in ecdf_pyrc.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    ec_temp_dict_pyrc = {"id":"EC_EComp","reaction":'Cgl0689:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    reaction = ec_temp_dict_pyrc['reaction']

    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()

    ec_temp_dict_pyrc['reaction'] = reaction_cleaned
    E_result_dict_pyrc[reaction_name] = ec_temp_dict_pyrc
for index, row in ecdf_aspk.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    ec_temp_dict_aspk = {"id":"EC_EComp","reaction":'Cgl0251:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = ec_temp_dict_aspk['reaction']

    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()

    ec_temp_dict_aspk['reaction'] = reaction_cleaned    
    E_result_dict_aspk[reaction_name] = ec_temp_dict_aspk

for index, row in ecdf_ipps.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    ec_temp_dict_ipps = {"id":"EC_EComp","reaction":'Cgl0248:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = ec_temp_dict_ipps['reaction']

    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()

    ec_temp_dict_ipps['reaction'] = reaction_cleaned    
    E_result_dict_ipps[reaction_name] = ec_temp_dict_ipps


for index, row in ecdf_gapd.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    ec_temp_dict_gapd = {"id":"EC_EComp","reaction":'Cgl0937:'+'g3p_c + nad_c + pi_c --> 13dpg_c + h_c + nadh_c' , "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = ec_temp_dict_gapd['reaction']
    ec_temp_dict_ipps['reaction'] = reaction_cleaned    
    E_result_dict_ipps[reaction_name] = ec_temp_dict_ipps

In [ ]:
edf_pyrc = lysine_SET[lysine_SET['reaction']=='PYRC']
edf_aspk = lysine_SET[lysine_SET['reaction']=='ASPK']
edf_ipps = lysine_SET[lysine_SET['reaction']=='IPPS']
edf_gapd = lysine_SET[lysine_SET['gene']=='Cgl0937']

In [ ]:
ET_result_dict_pyrc = {}
ET_result_dict_aspk = {}
ET_result_dict_ipps = {}
ET_result_dict_gapd = {}


for index, row in edf_pyrc.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    et_temp_dict_pyrc = {"id":"ET_EComp","reaction":'Cgl0689:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}

    reaction = et_temp_dict_pyrc['reaction']


    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()

    et_temp_dict_pyrc['reaction'] = reaction_cleaned
    ET_result_dict_pyrc[reaction_name] = et_temp_dict_pyrc
for index, row in edf_aspk.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    et_temp_dict_aspk = {"id":"ET_EComp","reaction":'Cgl0251:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = et_temp_dict_aspk['reaction']

    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()

    et_temp_dict_aspk['reaction'] = reaction_cleaned    
    ET_result_dict_aspk[reaction_name] = et_temp_dict_aspk

for index, row in edf_ipps.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    et_temp_dict_ipps = {"id":"ET_EComp","reaction":'Cgl0248:'+ equation, "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = et_temp_dict_ipps['reaction']

    reaction_cleaned = reaction.split('(', 1)[0].strip() + " : " + reaction.split(':', 1)[1].split(':', 1)[1].strip()


    et_temp_dict_ipps['reaction'] = reaction_cleaned    
    ET_result_dict_ipps[reaction_name] = et_temp_dict_ipps


for index, row in edf_gapd.iterrows():
    reaction_name = row['gene']
    equation = row['reaction_bio']
    flux_wild_values = [float(value.strip("'")) for value in row['enzyme_wild'].strip('[]').split(', ')]
    flux_over_values = [float(value.strip("'")) for value in row['enzyme_over'].strip('[]').split(', ')]
    et_temp_dict_gapd = {"id":"ET_EComp","reaction":'Cgl0937:'+'g3p_c + nad_c + pi_c --> 13dpg_c + h_c + nadh_c' , "wild_range": flux_wild_values, "eng_range": flux_over_values}
    reaction = et_temp_dict_gapd['reaction']



    et_temp_dict_ipps['reaction'] = reaction_cleaned    
    ET_result_dict_ipps[reaction_name] = et_temp_dict_ipps

## Case Study of Algorithm Advantages

In [ ]:
import matplotlib.pyplot as plt
import os


savepath = './figures'
os.makedirs(savepath, exist_ok=True)

data = [
    s_temp_dict_pyrc,
    T_result_dict_pyrc,
    ec_temp_dict_pyrc,
    et_temp_dict_pyrc
]


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (ax, item) in enumerate(zip(axes, data)):
    id = item['id']
    reaction = item['reaction']
    ranges = {
        'Reference': item['wild_range'],
        'Engineered': item['eng_range']
    }

    for i, (label, (start, end)) in enumerate(ranges.items()):
        start, end = float(start), float(end)
        ax.plot([i, i], [start, end], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [start, start], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [end, end], color='blue', linewidth=6)

    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels(list(ranges.keys()), fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    if idx < 2:
        ax.set_ylabel('Reaction Flux (mmol/gDW/h)', fontsize=12)
    else:
        ax.set_ylabel('Enzyme cost (g/gDW)', fontsize=12)

    ax.set_title(id + "\n" + reaction, fontsize=11)
    ax.grid(False)

plt.tight_layout()
plt.savefig(os.path.join(savepath, 'optforce_pyrc_4plots.svg'))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import os

savepath = './figures'
os.makedirs(savepath, exist_ok=True)


data = [
    s_temp_dict_aspk,
    T_result_dict_aspk,
    ec_temp_dict_aspk,
    et_temp_dict_aspk
]


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()  

for idx, (ax, item) in enumerate(zip(axes, data)):
    id = item['id']
    reaction = item['reaction']
    ranges = {
        'Reference': item['wild_range'],
        'Engineered': item['eng_range']
    }

    for i, (label, (start, end)) in enumerate(ranges.items()):
        start, end = float(start), float(end)
        ax.plot([i, i], [start, end], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [start, start], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [end, end], color='blue', linewidth=6)

    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels(list(ranges.keys()), fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    if idx < 2:
        ax.set_ylabel('Reaction Flux (mmol/gDW/h)', fontsize=12)
    else:
        ax.set_ylabel('Enzyme cost (g/gDW)', fontsize=12)

    ax.set_title(id + "\n" + reaction, fontsize=11)
    ax.grid(False)

plt.tight_layout()
plt.savefig(os.path.join(savepath, 'optforce_aspk_4plots.svg'))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import os


savepath = './figures'
os.makedirs(savepath, exist_ok=True)


data = [
    s_temp_dict_ipps,
    T_result_dict_ipps,
    ec_temp_dict_ipps,
    et_temp_dict_ipps
]


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten() 
for idx, (ax, item) in enumerate(zip(axes, data)):
    id = item['id']
    reaction = item['reaction']
    ranges = {
        'Reference': item['wild_range'],
        'Engineered': item['eng_range']
    }

    for i, (label, (start, end)) in enumerate(ranges.items()):
        start, end = float(start), float(end)
        ax.plot([i, i], [start, end], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [start, start], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [end, end], color='blue', linewidth=6)

    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels(list(ranges.keys()), fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    if idx < 2:
        ax.set_ylabel('Reaction Flux (mmol/gDW/h)', fontsize=12)
    else:
        ax.set_ylabel('Enzyme cost (g/gDW)', fontsize=12)

    ax.set_title(id + "\n" + reaction, fontsize=11)
    ax.grid(False)

plt.tight_layout()
plt.savefig(os.path.join(savepath, 'optforce_ipps_4plots.svg'))
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import os


savepath = './figures'
os.makedirs(savepath, exist_ok=True)


data = [
    s_temp_dict_gapd,
    T_result_dict_gapd,
    ec_temp_dict_gapd,
    et_temp_dict_gapd
]


fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()  

for idx, (ax, item) in enumerate(zip(axes, data)):
    id = item['id']
    reaction = item['reaction']
    ranges = {
        'Reference': item['wild_range'],
        'Engineered': item['eng_range']
    }

    for i, (label, (start, end)) in enumerate(ranges.items()):
        start, end = float(start), float(end)
        ax.plot([i, i], [start, end], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [start, start], color='blue', linewidth=6)
        ax.plot([i - 0.05, i + 0.05], [end, end], color='blue', linewidth=6)

    ax.set_xticks(range(len(ranges)))
    ax.set_xticklabels(list(ranges.keys()), fontsize=12)
    ax.tick_params(axis='y', labelsize=12)

    if idx < 2:
        ax.set_ylabel('Reaction Flux (mmol/gDW/h)', fontsize=12)
    else:
        ax.set_ylabel('Enzyme cost (g/gDW)', fontsize=12)

    ax.set_title(id + "\n" + reaction, fontsize=11)
    ax.grid(False)

plt.tight_layout()
plt.savefig(os.path.join(savepath, 'optforce_ipps_4plots.svg'))
plt.show()
